# 07 — Instrument closure: pin the probe, then re-sweep the realized grid

**BLIND-SAFE up to section 9. Run this BEFORE notebook 06.** Sections 1-8 score only the
**shape anchor** (the §5 gate factor, orthogonal to H1-H4) and the raw-pixel reference.
No targeted-factor value is read anywhere in this notebook.

## Why this notebook exists

The first calibration run (`results/calibration/calibration_shapes3d.raw.json`) is **void**.
It failed on three counts:

1. **A9 — budget confound.** The probe budget was fixed in *epochs*, so probe-train size and
   optimizer budget moved together (100 steps at n=2000 vs 1000 at n=40000). Headroom that
   appeared at small n could not be told apart from a probe that never converged.
2. **Degenerate extrapolation regime.** `make_value_holdout_splits` withholds whole *values*
   of the anchor. For a class label that leaves probe-test holding only unseen classes, so
   accuracy is 0 by construction: all 8 extrapolation cells returned the -1/3 zero-accuracy
   floor. That arm measured nothing.
3. **Selector artifact.** `recommended_config` filtered on headroom alone and sorted
   extrapolation first, so the regime that fails by construction — and is therefore never
   saturated — won automatically. Its verdict was a constant of the code.

All three are repaired in `src/`. This notebook re-runs the calibration under the repaired
instrument, pins the probe config on a criterion that both A8 §c and A8 §e must clear, and then
re-sweeps the three realized cells at that config.

## The re-sweep is mandatory regardless

The three existing stacks are stale against the current instrument: they carry four arrays
(`trained`, `random`, `perm`, `projector`) and are missing `random_projector` — the A8 §d
matched random-projector floor that H4 needs — and their `meta.json` has no `encoder_ckpts`
provenance. They must be re-swept whatever the calibration says, so pinning a probe config
first costs no extra compute.

## The decision rule, pre-committed here while blind

Read the branch off section 8 and follow it. Committing to this *before* seeing the numbers is
the point; choosing after would repeat the first run's error.

| Branch | Condition | Action |
|---|---|---|
| **A** | some `(regime=interpolation, n, steps)` clears §c (top-rung random floor < 0.90) **and** §e (\|gap\| <= 0.02) on **both** datasets | Pin it. Re-sweep (§9). Then unblind with notebook 06. |
| **B** | only `regime=composition` clears both | Stop. File amendment A10 changing the registered probe-test split, and extend `run_sweep` with the composition regime, before any sweep. |
| **C** | something clears §c but nothing clears §e | Do **not** pin on headroom alone — that is exactly the first run's error. Raise `--probe-steps` and re-run §6-§7. If the gap holds, invoke A9 consequence (2): report the linear rung as optimizer-limited and co-report every `Delta_G` against the closed-form rung. |
| **D** | nothing clears §c at any n, steps or regime | Saturation is a property of these datasets, not of the probe. Keep n=40000, run the confirmatory layer with A6(a)'s saturation gate active, and report the saturation itself as the finding plus the un-gated flip variants as diagnostics. Write it up as a pre-registered limitation, not a post-hoc discovery. |

**Setup:** Accelerator `GPU T4 x2`, Internet **On**.
**Outputs:** `results/calibration/calibration_{shapes3d,dsprites}.json`, then
`results/probes/{color,control,position}_strong/{stacks.npz,meta.json}`.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Datasets + image cache
`--build-cache` decompresses once into an uncompressed memmap the loaders mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

In [ ]:
# Restore prior probe/calibration OUTPUTS and encoder checkpoints so a timed-out
# session continues instead of recomputing.
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance"); INPUT = Path("/kaggle/input")
restored = 0
for src in list(INPUT.glob("*/results")) + list(INPUT.glob("*/probe-capacity-invariance/results")):
    for f in src.rglob("*"):
        if f.is_file() and f.suffix in (".npz", ".json", ".jsonl", ".pt"):
            dst = REPO / "results" / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst); restored += 1
print(f"restored {restored} prior result files")

## 5. Preflight — is the instrument actually repaired?

Fails loudly rather than burning a session on stale code. Checks the A9 step-budget pin,
the **A10 (a)** target-standardization repair (without it every continuous readout is void),
the **A10 (b)** all-factor headroom reporting, and the **A10 (c)** two-sided invariance rule.

In [ ]:
import inspect, json
from pathlib import Path
import numpy as np

from src.probes import ladder, instrument_calibration, instrument, hypotheses
from src.data import splits
from src.eval import metrics

src_cal = inspect.getsource(instrument_calibration.run)
checks = [
    ("A9  step budget pinned",        getattr(ladder, "DEFAULT_STEPS", None) == 1000, ""),
    ("A9  fit_rung accepts steps",    "steps" in inspect.signature(ladder.fit_rung).parameters, ""),
    ("A10a target standardization",   hasattr(ladder, "_standardize_targets"),
     "without this every CONTINUOUS readout is void (prereg A10 a)"),
    ("A10a biases exempt from wd",    "weight_decay\": 0.0" in inspect.getsource(ladder._train_one)
                                      or "weight_decay': 0.0" in inspect.getsource(ladder._train_one), ""),
    ("A10a cache version tag",        getattr(ladder, "INSTRUMENT_VERSION", None) == "a10",
     "resume caches from the broken ladder sit at the SAME (n, steps)"),
    ("A10b all-factor floors",        "all_factor_floors" in src_cal, ""),
    ("A10b pin needs every factor",   "all_factor_headroom_ok" in src_cal, ""),
    ("A10c epsilon_D",                hasattr(instrument, "epsilon_d"), ""),
    ("A10c two-sided flip",           hasattr(instrument, "deficit_flip_count"), ""),
    ("A10c H3 keys on eps_invariant", "eps_invariant" in inspect.getsource(hypotheses.analyze_cell), ""),
    ("compositional split available", hasattr(splits, "make_combination_holdout_splits"), ""),
]
bad = [n for n, ok, _ in checks if not ok]
for n, ok, why in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n}" + (f"   <- {why}" if (why and not ok) else ""))
assert not bad, f"stale code: {bad}. `git pull` in section 2, then restart the kernel."
print("\npreflight OK — instrument is at A10.")

## 6. Shapes3D calibration (repaired)

Two step budgets, so the A8 §e gap can be read as a function of optimizer budget rather than
assumed away: at n=40000 the *old* run already had 1000 steps and still showed a +0.065 gap, so
1000 alone is not known to be enough. `orientation` is the compositional partner because its arm
is excluded from the realized grid (A5), so the split touches no targeted factor.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.instrument_calibration \
    --dataset shapes3d --probe-train-sizes 2000 5000 10000 40000 \
    --probe-steps 1000 4000 --partner-factor orientation --n-holdout-partner 3 \
    --random-seed 0 1 2 --device cuda --num-workers 2 --out-root results/calibration

## 7. dSprites calibration (the position arm's dataset)

Partner is `scale`: `pos_x`/`pos_y` are the position arm's targeted factors, and dSprites
`orientation` is diagnostic-only per A3.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.instrument_calibration \
    --dataset dsprites --probe-train-sizes 2000 5000 10000 40000 \
    --probe-steps 1000 4000 --partner-factor scale --n-holdout-partner 2 \
    --random-seed 0 1 2 --device cuda --num-workers 2 --out-root results/calibration

## 8. The decision (A10 §b all-factor gate)

A config is pinnable only if it clears **all three** gates on **both** datasets:

* **A8 §c** — top-rung random-encoder floor < 0.90 on the **anchor**, so `G` has range to move in.
* **A10 §b** — *no factor at all* saturated at the top rung. A9's pin was validated on the
  categorical shape anchor alone, and the anchor is not representative: at n=40000 its floor is
  0.9858 while `floor_hue` and `wall_hue` reach 0.9982 and 0.9986. A pin that leaves every
  **targeted** readout saturated measures nothing.
* **A8 §e** — the ladder's Adam linear rung agrees with the convex solver to within 0.02, so
  `Delta_G` is a capacity difference and not an optimization difference.

This prints, per dataset and per candidate config, the **top-rung floor for every factor**,
`worst_top_rung_floor`, `saturated_factors_at_top`, and the linear-rung gap — then states whether
any **interpolation** config clears the all-factor gate.

**Three outcomes.**

* **A** - some *interpolation* config clears all three gates on both datasets. Pin it, run section 9.
* **B** - none does, but some *composition* config clears them. A10 §b's declared fallback is
  executable: file the composition-regime amendment, state that the estimand changes from
  lattice interpolation to compositional generalization, then re-pin and run section 9.
* **C** - neither regime clears it. The declared fallback fails the gate it was declared for,
  so it is not executable. File `lab-notebook/drafts/A14-headroom-gate-supersession-DRAFT.md`
  as a dated amendment and **commit it** - section 9 checks `prereg.md` for `### A14 -` after
  the `git pull` and refuses to sweep without it. A14 supersedes A10 §b's all-factor headroom
  *veto* with a per-readout *reporting* requirement (saturated readouts are routed through the
  A6 §a null-saturation gate, which already excludes them from H3 and the claim-quoted flip
  count) and pins on A8 §c + A8 §e alone. The estimand does **not** change.

Under every branch this must happen BEFORE any trained-encoder targeted-factor value is
computed.


In [ ]:
import json
from pathlib import Path

DATASETS = ("shapes3d", "dsprites")
runs = {}
for ds in DATASETS:
    p = Path(f"results/calibration/calibration_{ds}.json")
    if p.exists():
        runs[ds] = json.loads(p.read_text())
    else:
        print(f"{ds}: NOT RUN")

BRANCH = None
interp_ok = {}
for ds, r in runs.items():
    print(f"\n=== {ds} ===  partner={r.get('composition_partner')}  "
          f"saturation_level={r.get('saturation_level')}  tol={r.get('gap_tolerance')}")
    rows = [x for x in r["results"] if x["encoder_role"] == "random"]
    for x in rows:
        ff = x.get("factor_floor_mean", {})
        gates = []
        if x["saturated_at_top"]:            gates.append("A8c-anchor-saturated")
        if x.get("saturated_factors_at_top"): gates.append("A10b-saturated:" + ",".join(x["saturated_factors_at_top"]))
        if x["capacity_axis_flag"]:          gates.append(f"A8e-lin-gap={x['linear_rung_gap']:+.4f}")
        print(f"\n  {x['regime']:14s} n={x['probe_train_size']:<6d} steps={x['probe_steps']:<5d}"
              f"  worst_top_rung_floor={x.get('worst_top_rung_floor', float('nan')):.4f}"
              f"  lin_gap={x['linear_rung_gap']:+.4f}"
              f"  all_factor_headroom_ok={x.get('all_factor_headroom_ok')}")
        for name in sorted(ff):
            v = ff[name][-1]
            print(f"      {name:14s} top-rung floor {v:.4f}"
                  + ("   <- SATURATED" if v >= r.get("saturation_level", 0.90) else ""))
        print(f"      verdict: {'CLEARS ALL GATES' if not gates else ' | '.join(gates)}")

    clear = [x for x in rows if x["regime"] == "interpolation"
             and not x["saturated_at_top"] and not x["capacity_axis_flag"]
             and x.get("all_factor_headroom_ok")]
    interp_ok[ds] = clear
    print(f"\n  --> interpolation configs clearing the ALL-FACTOR gate: {len(clear)}")

# Composition is A10 (b)'s DECLARED fallback, so it is evaluated on the same gate rather
# than assumed to work. If it also fails, the fallback is unexecutable and branch C applies.
comp_ok = {}
for ds, r in runs.items():
    rows = [x for x in r["results"] if x["encoder_role"] == "random"]
    comp_ok[ds] = [x for x in rows if x["regime"] == "composition"
                   and not x["saturated_at_top"] and not x["capacity_axis_flag"]
                   and x.get("all_factor_headroom_ok")]
    print(f"  --> {ds}: composition configs clearing the ALL-FACTOR gate: {len(comp_ok[ds])}")

# A14 (g): pinnable = A8 (c) on the anchor AND A8 (e) linear-rung agreement, on BOTH
# datasets. The all-factor floor table is still computed and is reported per readout.
A8_ok = {ds: {(x["probe_train_size"], x["probe_steps"])
              for x in r["results"]
              if x["encoder_role"] == "random" and x["regime"] == "interpolation"
              and not x["saturated_at_top"] and not x["capacity_axis_flag"]}
         for ds, r in runs.items()}
shared = sorted(set.intersection(*A8_ok.values()), key=lambda k: (-k[0], k[1])) if A8_ok else []

PROBE_TRAIN = PROBE_STEPS = REGIME = None
AMENDMENT_FILED = "### A14 \u2014" in Path("preregistration/prereg.md").read_text()

if runs and all(interp_ok.get(ds) for ds in runs):
    BRANCH = "A"
    pick = sorted(interp_ok[list(runs)[0]],
                  key=lambda x: (-x["probe_train_size"], x["probe_steps"]))[0]
    PROBE_TRAIN, PROBE_STEPS, REGIME = pick["probe_train_size"], pick["probe_steps"], "interpolation"
    print("\nBRANCH A - an interpolation config clears the all-factor gate on every dataset.")
elif runs and all(comp_ok.get(ds) for ds in runs):
    BRANCH = "B"
    print("\nBRANCH B - no interpolation config clears the A10 (b) all-factor gate, but the\n"
          "declared COMPOSITION fallback does. STOP. Do not run section 9 yet.\n"
          "  1. File the composition-regime amendment, filling in the trigger numbers above.\n"
          "  2. State explicitly that the ESTIMAND changes from lattice interpolation to\n"
          "     compositional generalization, and that no number under the new regime is\n"
          "     comparable to any interpolation number in A4/A8/A10.\n"
          "  3. Only then re-pin from the composition rows and run section 9.")
elif runs:
    BRANCH = "C"
    print("\nBRANCH C - NEITHER regime clears the A10 (b) all-factor gate. A10 (b)'s declared\n"
          "composition fallback fails the gate it was declared for, so it is not executable:\n"
          "adopting it would change the estimand AND still fail. The estimand does not change.\n")
    if not shared:
        print("  No configuration clears A8 (c) + A8 (e) on both datasets either. STOP and\n"
              "  re-derive the instrument; do not sweep.")
    else:
        PROBE_TRAIN, PROBE_STEPS = shared[0]
        REGIME = "interpolation"
        print(f"  Pinnable on A8 (c) + A8 (e), both datasets: {shared}\n"
              f"  A14 (g) pin -> probe_train={PROBE_TRAIN} steps={PROBE_STEPS} regime={REGIME}\n"
              "  Required before section 9:\n"
              "  1. File lab-notebook/drafts/A14-headroom-gate-supersession-DRAFT.md as a dated\n"
              "     amendment in preregistration/prereg.md, with the trigger numbers above.\n"
              "  2. COMMIT AND PUSH it. Section 9 checks prereg.md for the A14 header after the\n"
              "     git pull and refuses to sweep without it.\n"
              "  3. Every saturated readout above is routed through the A6 (a) null-saturation\n"
              "     gate: out of H3's confirmed set, out of the claim-quoted flip count, read as\n"
              "     suppression or no-headroom - never as genuine invariance.\n"
              f"  amendment filed: {AMENDMENT_FILED}")

print(f"\nBRANCH={BRANCH}  PROBE_TRAIN={PROBE_TRAIN}  PROBE_STEPS={PROBE_STEPS}  REGIME={REGIME}")


> **A10 (a): the existing probe stacks are VOID.** Every continuous-factor entry in
> `results/probes/*/stacks.npz` was computed with the unstandardized-target ladder and must be
> recomputed, not resumed. `run_sweep`'s resume cache is keyed on `INSTRUMENT_VERSION`, so an
> `a10` run cannot reload a pre-A10 cache even at a matching `(n, steps)` — but if you restored
> old outputs in section 4, delete `results/probes/*/stacks.npz` before sweeping so a stale
> stack cannot be mistaken for a fresh one.

In [ ]:
from pathlib import Path
for f in Path("results/probes").glob("*/stacks.npz"):
    print("removing void pre-A10 stack:", f); f.unlink()
for f in Path("results/probes").glob("*/_cache/n*"):   # untagged = pre-A10
    print("stale untagged cache:", f)

## 9. Re-sweep the three realized cells at the pinned config

**This is the last blind-safe section.** The sweep fits every factor, but nothing here prints a
targeted-factor value — it writes `stacks.npz` and reports only the shape-gate summary. Read
`results/hypotheses/` (notebook 06) only after this completes.

`control_strong` is gate-exempt by design (A2): a minimal-augmentation encoder is *expected* to
sit at the random floor, so 0/12 passing is the intended datum, not a failure.

In [ ]:
from pathlib import Path

assert BRANCH in ("A", "C"), (
    f"branch {BRANCH}: do not sweep. Section 8 printed what to do instead - sweeping here "
    "would pin a probe config the calibration did not license."
)
if BRANCH == "C":
    # A14 supersedes A10 (b)'s pinning veto. It is only in force once filed, and it must be
    # filed BEFORE the data it licenses exists. This is the enforcement of that ordering.
    assert AMENDMENT_FILED, (
        "branch C requires amendment A14 filed in preregistration/prereg.md and pushed "
        "(the git pull in section 2 is what brings it here). Draft: "
        "lab-notebook/drafts/A14-headroom-gate-supersession-DRAFT.md"
    )
assert PROBE_TRAIN and PROBE_STEPS and REGIME, "section 8 did not pin a probe configuration"

# run_sweep only WARNS when it gets too few encoders, so an unresolved glob would
# silently sweep zero trained encoders and write a stack that looks valid. Fail here.
CELLS = {
    "color_strong": "results/encoders/color_strong_seed*/backbone.pt",
    "control_strong": "results/encoders/control_strong_seed*/backbone.pt",
    "position_strong": "results/encoders/position_strong_seed*/backbone.pt",
}
for cell, pattern in CELLS.items():
    found = sorted(Path().glob(pattern))
    print(f"{cell:16s} {len(found):>2d} checkpoints")
    assert len(found) == 12, (
        f"{cell}: '{pattern}' resolved {len(found)} checkpoints, expected 12. The restored "
        "input layout differs from the training layout — fix the glob before sweeping."
    )

SEEDS = "0 1 2 3 4 5 6 7 8 9 10 11"
print(f"\nsweeping at probe_train={PROBE_TRAIN} probe_steps={PROBE_STEPS} regime={REGIME}")

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition color --strength strong \
    --encoders results/encoders/color_strong_seed*/backbone.pt \
    --random-seed {SEEDS} --subsample {PROBE_TRAIN} --probe-steps {PROBE_STEPS} \
    --device cuda --num-workers 2 --resume --out-root results/probes

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition control --strength strong \
    --encoders results/encoders/control_strong_seed*/backbone.pt \
    --random-seed {SEEDS} --subsample {PROBE_TRAIN} --probe-steps {PROBE_STEPS} \
    --device cuda --num-workers 2 --resume --out-root results/probes

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset dsprites --condition position --strength strong \
    --encoders results/encoders/position_strong_seed*/backbone.pt \
    --random-seed {SEEDS} --subsample {PROBE_TRAIN} --probe-steps {PROBE_STEPS} \
    --device cuda --num-workers 2 --resume --out-root results/probes

## 10. Verify the contract artifacts

Every cell must carry the A8 §d random-projector floor, checkpoint provenance, and the pinned
probe config. A cell that fails here is not usable by notebook 06.

In [ ]:
import json
import numpy as np
from pathlib import Path

ok = True
for cell in ("color_strong", "control_strong", "position_strong"):
    d = Path("results/probes") / cell
    if not (d / "stacks.npz").exists():
        print(f"{cell:16s} NOT SWEPT"); ok = False; continue
    z = np.load(d / "stacks.npz"); m = json.loads((d / "meta.json").read_text())
    problems = []
    if "random_projector" not in z.files: problems.append("no random_projector (A8 §d)")
    if not m.get("encoder_ckpts"): problems.append("no checkpoint provenance")
    if m.get("probe_train_size") != PROBE_TRAIN: problems.append(f"n={m.get('probe_train_size')}")
    if m.get("probe_steps") != PROBE_STEPS: problems.append(f"steps={m.get('probe_steps')}")
    g = m["quality_gate"]
    print(f"{cell:16s} {'OK  ' if not problems else 'BAD '} "
          f"seeds={z['trained'].shape[0]} gate={g['n_passed']}/{g['n_encoders']} "
          f"{'; '.join(problems)}")
    ok &= not problems
print("\nready for notebook 06" if ok else "\nNOT ready — fix the above before unblinding")

## 11. Persist for the next session
`/kaggle/working` is the notebook output. Click **Save Version** when this finishes, then
**Add Input -> this output** on the next run to resume.

In [ ]:
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")